# RAG Branching Example

A graph that answers questions about this system using RAG on `vector_db/llm_graph_docs`.

**Graph layout:**

```
branch_llm  ──►  check_type (ConditionalNode)
                    │
          ┌─────────┴──────────┐
       coding               general
          │                    │
  coding_retrieval    general_retrieval
          │                    │
    coding_llm          general_llm
```

- **coding** branch: retrieves relevant docs and answers with a code-focused prompt
- **general** branch: retrieves relevant docs and answers with a conceptual explanation prompt

In [1]:
from dotenv import load_dotenv
import os
from IPython.display import Markdown
from openai import OpenAI

from llm_graph.llm.response_functions import OpenAI_response_fn
from llm_graph.factories.llm import create_llm_node
from llm_graph.factories.rag import create_rag_query_pair
from llm_graph.core.nodes import ConditionalNode
from llm_graph.core.graphrunner import GraphRunner
from llm_graph.core.sessionrunner import SessionRunner

In [2]:
load_dotenv()
client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ.get("NVIDIA_API_KEY"),
    timeout=30.0
)
response_fn = OpenAI_response_fn(client, model="qwen/qwen3-coder-480b-a35b-instruct")

branch_response_fn = OpenAI_response_fn(
    client=client,
    model="meta/llama-3.3-70b-instruct"
)

## Prompts

In [9]:
branching_prompt = """
You are making a decision on the type of query given.
This will either be a coding query (asking for code examples, implementation details, usage snippets)
or a general query (asking for conceptual explanations, descriptions, or comparisons).

Respond ONLY with valid JSON with key 'query_type' set to either 'coding' or 'general'.

The query is: {user_query}
"""

coding_prompt = """
You are an expert on the llm-graph-engine system.
A user has asked a coding question about this system.
Use the retrieved documentation to give a clear, complete code example that answers the query.
Include imports where relevant. Do not go beyond what is asked.

Respond ONLY with valid JSON with key 'answer' containing your response.

The query is: {user_query}
"""

general_prompt = """
You are an expert on the llm-graph-engine system.
A user has asked a general question about this system.
Use the retrieved documentation to give a clear conceptual explanation.
Your response should be in markdown, using bullet points, numbered lists, and headers as appropriate.
No sample code should be generated for answers to general questions.
Respond ONLY with valid JSON with key 'answer' containing your response.
 
The query is: {user_query}
"""

## Build the graph

In [10]:
# Classifies the query — routes to 'coding' or 'general'
branch_node = create_llm_node(
    response_fn=branch_response_fn,
    name="branch_llm",
    prompt_template=branching_prompt,
    next_node_name="check_type",
)

check_type_node = ConditionalNode(
    name="check_type",
    condition_fn=lambda state: state["query_type"],
)

In [11]:
DB_PATH = "../vector_db"
COLLECTION = "llm_graph_docs"

# Coding branch: retrieval → LLM with coding prompt
coding_nodes = create_rag_query_pair(
    path=DB_PATH,
    collection_name=COLLECTION,
    response_fn=response_fn,
    retrieval_node_name="coding",
    llm_node_name="coding_llm",
    prompt_template=coding_prompt,
)

# General branch: retrieval → LLM with general prompt
general_nodes = create_rag_query_pair(
    path=DB_PATH,
    collection_name=COLLECTION,
    response_fn=response_fn,
    retrieval_node_name="general",
    llm_node_name="general_llm",
    prompt_template=general_prompt,
)

In [12]:
graphrunner = GraphRunner.build(
    node_dicts=[
        {"branch_llm": branch_node, "check_type": check_type_node},
        coding_nodes,
        general_nodes,
    ],
    start_node="branch_llm",
)

session = SessionRunner(
    graph=graphrunner,
    session_keys=["message_history"],
)

## Run some queries

In [13]:
# General question — should route to the 'general' branch
r1 = session.execute({"user_query": "How does GraphRunner.build() work and when would I use it instead of the regular constructor?"})
Markdown(r1["state_dict"]["answer"])

# GraphRunner.build() vs Regular Constructor

## How GraphRunner.build() Works

The `GraphRunner.build()` method is a class method that provides an alternative way to construct a GraphRunner instance with the following characteristics:

- **Factory Pattern**: It's designed as a factory method that can create complex GraphRunner configurations
- **Pre-configuration**: It accepts pre-built node dictionaries and state configurations
- **Workflow Assembly**: It's specifically intended for assembling complete workflows from component pieces

## When to Use GraphRunner.build()

Use `GraphRunner.build()` instead of the regular constructor when:

1. **Component-Based Construction**: You're building workflows using the factory functions (like those in `llm_graph.factories.tool`) that create related node pairs

2. **Complex Workflow Assembly**: You need to assemble multiple interconnected nodes that work together as a unit

3. **Pre-built State Integration**: You have pre-existing node dictionaries and state configurations that need to be integrated

4. **Tool Workflow Patterns**: You're implementing structured patterns like the tool workflow architecture shown in the documentation

## Key Differences

- **Regular Constructor**: Use for simple, direct instantiation with basic parameters
- **GraphRunner.build()**: Use for complex workflow assembly where multiple nodes and their relationships need to be established programmatically

The `build()` method is particularly valuable when working with the conditional node patterns and tool workflow structures described in the architecture documentation.

In [14]:
# Coding question — should route to the 'coding' branch
r2 = session.execute({"user_query": "How do I use create_rag_query_pair to build a RAG workflow?"})
Markdown(r2["state_dict"]["answer"])

## Using create_rag_query_pair to Build a RAG Workflow

The `create_rag_query_pair` function simplifies RAG workflow creation by generating a pre-wired retrieval and LLM pair. Here's how to use it:

### Basic Usage

```python
from openai import OpenAI
from llm_graph.llm.response_functions import OpenAI_response_fn
from llm_graph.factories.rag import create_rag_query_pair
from llm_graph.core.graphrunner import GraphRunner

# Initialize OpenAI client
client = OpenAI()
response_fn = OpenAI_response_fn(client=client)

# Create the RAG node pair
rag_nodes = create_rag_query_pair(
    path="vector_db",
    collection_name="my_docs",
    response_fn=response_fn,
    llm_node_name="llm",
)

# Build the GraphRunner
runner = GraphRunner.build(
    node_dicts=[rag_nodes],
    start_node="retrieval",
)

# Execute the workflow
output = runner.execute({"user_query": "How does GraphRunner work?"})
print(output["state_dict"]["answer"])
```

### With Custom Prompt Template

```python
# Custom prompt template (note that {retrieved_context} is injected automatically)
custom_prompt = """
You are a helpful assistant. Use the provided context to answer the question.
Respond in JSON with key 'answer'.

The query is: {user_query}
"""

rag_nodes = create_rag_query_pair(
    path="vector_db",
    collection_name="my_docs",
    response_fn=response_fn,
    llm_node_name="llm",
    prompt_template=custom_prompt,  # {retrieved_context} will be injected automatically
)

runner = GraphRunner.build(
    node_dicts=[rag_nodes],
    start_node="retrieval",
)

output = runner.execute({"user_query": "Explain RAG implementation"})
```

### Key Points

- The function creates both retrieval and LLM nodes wired together
- Both nodes default to using `"user_query"` as the query key in the state dictionary
- Use `GraphRunner.build()` to assemble the workflow
- The workflow expects a `user_query` key in the input state dictionary
- Make sure your ChromaDB collection is populated using `scripts/build_rag_index.py` before execution

In [9]:
# Another general question
r3 = session.execute({"user_query": "What are the parse-error and tool-error retry systems in the tool factory, and how do they relate to each other?"})
Markdown(r3["state_dict"]["answer"])

The parse-error and tool-error retry systems in the tool factory are two complementary error handling mechanisms that work together to create robust tool workflows. The parse-error retry system handles cases where the LLM generates malformed JSON output. When this happens, a ConditionalNode detects the parse error and routes execution to a retry LLM node that gets a prompt including the bad output and error message, creating a loop until valid JSON is produced. The tool-error retry system handles cases where the tool execution itself fails (either by raising an exception or returning a success=False status). When this occurs, a ConditionalNode checks for tool success and routes to either the tool analysis node (on success) or to a retry LLM that can generate new arguments for the tool. These systems relate to each other by being part of a unified error handling pipeline - the tool-error retry system actually routes through the parse-error checking mechanism, meaning that if a retry is triggered due to tool failure, the subsequent LLM output parsing is still protected by the parse-error retry system. This creates a comprehensive retry mechanism where both LLM parsing issues and tool execution failures are automatically handled.

In [11]:
# Another coding question
r4 = session.execute({"user_query": "Show me how to use the model parameter in the @tool_call decorator with Pydantic validation."})
Markdown(r4["state_dict"]["answer"])

```python
from pydantic import BaseModel, Field

from llm_graph.tools.tool_call import tool_call

# 1) Define a Pydantic model that describes/validates the tool arguments
class SearchArgs(BaseModel):
    query: str = Field(..., min_length=1)
    n_results: int = Field(3, ge=1, le=10)

# 2) Decorate your function with @tool_call and pass model=...
#    - The runtime will read kwargs from state["search_params"]
#    - It will validate them against SearchArgs before calling the function
#    - The return value will be written to state["search_results"] in the delta
@tool_call(input_key="search_params", output_key="search_results", model=SearchArgs)
def search_database(query: str, n_results: int = 3):
    # ... your real retrieval logic here ...
    return {"hits": [f"Result {i} for '{query}'" for i in range(1, n_results + 1)]}

# Example: calling the decorated tool with a state dict
state = {
    "search_params": {
        "query": "GraphRunner.build",
        "n_results": 3,
    }
}

# The decorated function is called with the whole state dict
delta = search_database(state)
print(delta["search_results"])  # {'hits': [...]} 

# Example: invalid args fail validation (returns a failure result instead of raising)
bad_state = {"search_params": {"query": "", "n_results": 999}}
failed_delta = search_database(bad_state)
print(failed_delta)  # will contain a failure result under output_key (implementation-defined)

# (Optional) tool metadata attached by the decorator (used by factories)
print(search_database.tool_meta["input_key"])      # 'search_params'
print(search_database.tool_meta["output_key"])     # 'search_results'
print(search_database.tool_meta["schema_model"])   # <class '__main__.SearchArgs'>
```


In [10]:
# Another coding question
r4 = session.execute({"user_query": "Show me how to use the model parameter in the @tool_call decorator with Pydantic validation."})
Markdown(r4["state_dict"]["answer"])

Here's how to use the model parameter in the @tool_call decorator with Pydantic validation:

```python
from pydantic import BaseModel
from llm_graph.tools import tool_call

class SearchArgs(BaseModel):
    query: str
    n_results: int = 3
    category: str = "general"

@tool_call(input_key="search_params", output_key="search_results", model=SearchArgs)
def search_database(query: str, n_results: int = 3, category: str = "general"):
    # Your search implementation here
    return f"Found {n_results} results for query: {query} in category: {category}"

# The decorator will automatically validate the input arguments
# against the SearchArgs model before calling the function

# Example usage in a workflow:
# When the state dict contains:
# {"search_params": {"query": "python tools", "n_results": 5}}
# The tool will execute successfully

# If invalid data is passed:
# {"search_params": {"query": "python tools", "invalid_field": "test"}}
# The validation will fail and return a failure result instead of raising an exception

# You can also access the tool metadata:
print(search_database.tool_meta)
# This includes: input_key, output_key, schema_model, tool_name, tool_doc, tool_signature
```

## Inspect the trace

In [11]:
# Show the execution path for the last query
graphrunner.print_trace()

Step 1
  Node: branch_llm
  Node type: FunctionalNode
  Input: {'message_history': [{'role': 'user', 'content': "\nYou are making a decision on the type of query given.\nThis will either be a coding query (asking for code examples, implementation details, usage snippets)\nor a general query (asking for conceptual explanations, descriptions, or comparisons).\n\nRespond ONLY with valid JSON with key 'query_type' set to either 'coding' or 'general'.\n\nThe query is: How does GraphRunner.build() work and when would I use it instead of the regular constructor?\n"}, {'role': 'assistant', 'content': '{"query_type": "general"}'}, {'role': 'user', 'content': "\nUse the following retrieved context to answer the question.\nContext :\nComponent documentation: GraphRunner\n\n### Component: GraphRunner\n\nPurpose:\nGraphRunner manages execution of workflow graphs by controlling node traversal\nand maintaining runtime state. It also tracks token usage and enforces optional node visit limits. It does 

In [12]:
# Token usage across the whole session
print(f"In tokens : {session.in_tokens}")
print(f"Out tokens: {session.out_tokens}")

In tokens : 16812
Out tokens: 1116
